In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [14]:
%pip install "pillow>=10" "matplotlib>=3.9"

In [15]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', 'torch==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cpu',
    ])

import torch
from PIL import Image
import matplotlib

print('PyTorch:', torch.__version__)
print('이미지·그래프 패키지:', Image.__version__, matplotlib.__version__)
probe = torch.tensor([1.0, 2.0], device='cpu')
assert torch.equal(probe * 2, torch.tensor([2.0, 4.0]))
print('CPU 텐서 연산 확인 완료')

PyTorch: 2.14.1+cpu
이미지·그래프 패키지: 11.3.0 3.10.0
CPU 텐서 연산 확인 완료


In [16]:
import sys
from importlib.metadata import version, requires
from packaging.requirements import Requirement
from packaging.markers import default_environment

import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_algorithms
import qiskit_machine_learning
import qiskit_optimization
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

expected = {
    'qiskit': '2.2.3',
    'qiskit-machine-learning': '0.9.0',
    'qiskit-optimization': '0.7.0',
    'qiskit-algorithms': '0.4.0',
}
print('Python:', sys.version.split()[0])
environment = default_environment()
environment['extra'] = ''
for package, target in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == target, f'{package} 설치 셀을 다시 실행하세요.'
    for specification in requires(package) or []:
        requirement = Requirement(specification)
        if requirement.marker and not requirement.marker.evaluate(environment):
            continue
        actual = version(requirement.name)
        assert actual in requirement.specifier, f'의존성 확인: {specification}, 설치 {actual}'
for package in ('numpy', 'scipy', 'scikit-learn'):
    print(package, version(package))
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
probability = Statevector.from_instruction(qc).probabilities()
assert np.allclose(probability, [0.5, 0, 0, 0.5])
print('회로 확률:', probability)
print('패키지 의존성·import·회로 연산 확인 완료')

Python: 3.13.18
qiskit 2.2.3
qiskit-machine-learning 0.9.0
qiskit-optimization 0.7.0
qiskit-algorithms 0.4.0
numpy 2.4.6
scipy 1.16.3
scikit-learn 1.6.1
회로 확률: [0.5 0.  0.  0.5]
패키지 의존성·import·회로 연산 확인 완료


In [17]:
import numpy as np

logits = np.log(np.array([[0.70, 0.20, 0.10], [0.40, 0.35, 0.25]]))
shifted = logits - logits.max(axis=1, keepdims=True)
probability = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
threshold = 0.6
confidence = probability.max(axis=1)
base = probability.argmax(axis=1)
route = confidence < threshold
qml_prediction = np.array([2])
final = base.copy()
final[route] = qml_prediction
print('확률:', probability)
print('신뢰도:', confidence, '분기:', route)
print('기본 예측:', base, '최종 예측:', final)
assert np.array_equal(final, [0, 2])

확률: [[0.7  0.2  0.1 ]
 [0.4  0.35 0.25]]
신뢰도: [0.7 0.4] 분기: [False  True]
기본 예측: [0 0] 최종 예측: [0 2]


In [18]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel

training = np.array([[0.1,0.2,0.3], [0.4,0.3,0.6], [0.7,0.8,0.5], [0.9,0.6,0.8]])
testing = np.array([[0.2,0.3,0.4]])
pca = PCA(n_components=2, svd_solver='full')
scaler = MinMaxScaler(feature_range=(0, np.pi))
train_angles = scaler.fit_transform(pca.fit_transform(training))
test_angles = np.clip(scaler.transform(pca.transform(testing)), 0, np.pi)
kernel = FidelityStatevectorKernel(feature_map=zz_feature_map(2, reps=2))
K_train = kernel.evaluate(train_angles)
K_test = kernel.evaluate(test_angles, train_angles)
print('학습 커널:', K_train.shape, '시험 커널:', K_test.shape)
print('대각 성분:', np.round(np.diag(K_train), 6))
print('최소 고유값:', round(float(np.linalg.eigvalsh(K_train).min()), 6))
assert np.allclose(K_train, K_train.T)
assert np.allclose(np.diag(K_train), 1.0)
assert K_test.shape == (1, 4)

학습 커널: (4, 4) 시험 커널: (1, 4)
대각 성분: [1. 1. 1. 1.]
최소 고유값: 0.130337


최종 예측 배열은 기본 모델 예측으로 시작합니다. 신뢰도가 임계값 미만이고 QSVC가 학습된 행만 보완 결과로 교체합니다. 전체 시험 정확도와 macro F1을 계산하고, 기본 모델의 오분류를 수정한 건수와 기본 정답을 오분류로 바꾼 건수를 따로 출력합니다. 두 건수의 차이는 전체 정답 수의 증감과 일치해야 합니다.

모델 상태·PCA·스케일러·QSVC·임계값·클래스 순서를 저장합니다. 저장 파일을 다시 읽은 뒤 동일한 시험 입력의 예측이 일치하는지 확인합니다. 학습 곡선은 흰 배경의 PNG로 저장합니다.